Building a Simple Chatbot with LangGraph

LangGraph makes it easy to build structured, stateful applications like chatbots. In this example we’ll learn how to create a basic chatbot that can classify user input as either a greet, search query and respond accordingly.

*Step 1: Install the Dependencies

Installs the required dependencies,

langgraph: Framework for building graph-based AI workflows.
langchain: Popular toolkit for LLM-powered AI applications.
google-generativeai: Google’s API for Generative AI (Gemini models).

In [ ]:
!pip install langgraph langchain google-genai

Step 2: Setup Gemini API

Imports the Google Generative AI Python SDK.

Configures the API with our private key for authentication.

Initializes the Gemini 1.5 Flash model for fast, multimodal LLM responses.

Defines an ask_gemini function that takes a prompt (user question) and generates a response from Gemini and handles errors gracefully by returning an apologetic message if the API fails.

In [ ]:
# setup API

from google.colab import userdata
from google import genai

key = userdata.get('GEMINI_API_KEY')
print("Key loaded:", key is not None)

client = genai.Client(api_key=key)

In [ ]:
# Define a Function
def ask_gemini(prompt: str) -> str:
  try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text
  except Exception as e:
    print(f"Error: {e}")
    return "Sorry, something went wrong with the Gemini API."

Step 3: Define Chatbot State

We will import Optional and TypedDict for strict type checking and creates a GraphState type:

Holds the current question, its classification (greeting/search) and the final response.

Ensures clarity and structure in state handling during workflow execution.

In [ ]:
from typing import Optional
from typing_extensions import TypedDict

class GraphState(TypedDict):
    question: Optional[str]
    classification: Optional[str]
    response: Optional[str]


Step 4: Classify Input

We define classify, which takes the workflow state and analyzes the user's question.

Checks if the question is a greeting. For example keywords like hi, hello, etc.
Tags the question as either "greeting" or "search" for branching logic later.
Returns the updated state with the new classification.




In [ ]:
def Classify(state: GraphState) -> GraphState:
  question = state.get("question", "").lower()
  if any(word in question for word in ["hello", "hi", "hy", "good morning", "good evening","Hy","Hello","good afternoon"]):
    classification = "greetings"
  else:
    classification = 'search'
  return {
      **state,
      "classification": classification
  }

Step 5: Respond Using Gemini (or Greeting)

This define respond which generates appropriate output based on classification.

For greetings, returns a friendly welcome message.

For search questions, calls Gemini via ask_gemini and fetches an AI-generated answer.

Handles unknown classifications with a safety fallback response.
Updates and returns the state with the generated reply.

In [ ]:
def respond(state: GraphState) -> GraphState:
  classification = state.get("classification")
  question = state.get("question")

  if classification == "greetings":
    response = "Hello! What's on Your Mind Today!"
  elif classification == "search":
    response = ask_gemini(question)
  else:
    response = "I'm not sure how to respond to that."

  return {
      **state,
      'response': response
  }


Step 6: Build LangGraph Workflow

Import tools for network graph creation and visualization.

Build the workflow graph using LangGraph, adding nodes for classification and response, connecting them with edges and compiling the app.

Include a function to visually display the workflow using networkx and matplotlib, aiding understanding and troubleshooting.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)
builder.add_node("classify", Classify)
builder.add_node("respond", respond)
builder.set_entry_point("classify")
builder.add_edge("classify", "respond")
builder.set_finish_point("respond")
app = builder.compile()

def visualize_workflow(builder):
    G = nx.DiGraph()

    for node in builder.nodes:
        G.add_node(node)
    for edge in builder.edges:
        G.add_edge(edge[0], edge[1])

    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_size=3000,
            node_color="skyblue", font_size=12, font_weight="bold", arrows=True)

    plt.title("Langchain Workflow Visualization")
    plt.show()

visualize_workflow(builder)










Step 7: Interactive Chat Interface

Create a command-line chatbot that processes user inputs until “exit” or “quit” is typed.

Send each input through the workflow graph and returns the bot’s response, either a greeting or an AI-powered answer.

In [ ]:

print("=== Gemini-Powered Chatbot ===")
print("Type your question below. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("Bot: Goodbye!")
        break

    state = {"question": user_input}
    result = app.invoke(state)
    print("Bot:", result["response"])